# ICL1D+: Contrastive In-Context Learning Baseline

**Paper:** Li et al. (COLING 2025) — "Enhancing RAG: A Study of Best Practices"

**Config:** ICL1D+ — retrieve 1 similar question from KB, include correct + incorrect answer as contrastive example

**Model:** Mistral-7B-Instruct-v0.2 (fp16, local)

**Datasets:** TruthfulQA (100Q test / 615Q KB) · MMLU (285Q test / 1539Q KB, seed=42, zero overlap)

**Paper results (fp16, full dataset):**
| Dataset | R1 | R2 | RL | ECS | MAUVE |
|---------|----|----|-----|-----|-------|
| TQA     | 30.62 | 17.45 | 27.79 | 58.96 | 73.86 |
| MMLU    | 26.01 | 17.46 | 24.90 | 47.04 | 37.24 |

In [1]:
# ── Cell 1: Environment check ────────────────────────────────────────────────
import sys, torch, importlib

required = ['transformers','sentence_transformers','faiss','datasets',
            'rouge_score','mauve','sklearn','tqdm','bitsandbytes','accelerate']
missing = [p for p in required if not importlib.util.find_spec(p)]
if missing: print('MISSING:', missing)
else: print('All packages OK')

print(f'Python: {sys.version[:6]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

All packages OK
Python: 3.10.2
PyTorch: 2.5.1+cu121
CUDA: True | GPU: Quadro RTX 8000
VRAM: 47.5 GB


In [2]:
# ── Cell 2: Config ───────────────────────────────────────────────────────────
import os, sys

BASE_DIR     = '/home/user/RAG_Best_Practices/RAG_best_practices-main'
MODELS_DIR   = '/home/user/RAG_Best_Practices/models'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUTPUT_DIR   = '/home/user/RAG_Best_Practices/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(BASE_DIR)
sys.path.insert(0, BASE_DIR)

# DEV MODE  : N_TQA=100, MMLU_PER_SUBJECT=5  → ~35min
# PAPER MODE: N_TQA=817, MMLU_PER_SUBJECT=32 → ~4h
N_TRUTHFULQA     = 100
MMLU_PER_SUBJECT = 5
RANDOM_SEED      = 42
QUANT            = None   # None=fp16, '4bit' for low VRAM

# ICL1D+ params (from paper)
ICL_PARAMS = {
    'top_k_docs':      1,     # retrieve 1 similar question from KB
    'max_gen_tokens':  25,
    'num_beams':       2,
}

print('Config OK')
print(f'TruthfulQA: {N_TRUTHFULQA}Q | MMLU: {MMLU_PER_SUBJECT}x57={MMLU_PER_SUBJECT*57}Q | seed={RANDOM_SEED}')
print(f'ICL1D+ config: top_k_docs={ICL_PARAMS["top_k_docs"]} (1 contrastive example per question)')

Config OK
TruthfulQA: 100Q | MMLU: 5x57=285Q | seed=42
ICL1D+ config: top_k_docs=1 (1 contrastive example per question)


In [3]:
# ── Cell 3: Load datasets (zero-overlap train/test split) ────────────────────
import gc, numpy as np, pandas as pd
from datasets import load_from_disk

CHOICE_LABELS = ['A', 'B', 'C', 'D']

# ── TruthfulQA
print('Loading TruthfulQA...')
tqa_raw = load_from_disk(f'{DATASETS_DIR}/truthfulqa').to_pandas()
tqa_all = tqa_raw[['question','best_answer','correct_answers','incorrect_answers']].copy()
tqa_all['correct_answers']   = tqa_all['correct_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['incorrect_answers'] = tqa_all['incorrect_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['best_answer']       = tqa_all['best_answer'].apply(lambda x: [x] if x else [])
tqa_all = tqa_all[(tqa_all['correct_answers'].apply(len) > 1) &
                   (tqa_all['incorrect_answers'].apply(len) > 1)].reset_index(drop=True)

tqa_test_idx = tqa_all.sample(n=N_TRUTHFULQA, random_state=RANDOM_SEED).index
tqa     = tqa_all.loc[tqa_test_idx].reset_index(drop=True)    # test set
tqa_icl = tqa_all.drop(tqa_test_idx).reset_index(drop=True)   # ICL KB (no overlap)
del tqa_raw; gc.collect()
print(f'  test={len(tqa)}Q | KB={len(tqa_icl)}Q (zero overlap)')

# ── MMLU (stratified)
print('Loading MMLU...')
mmlu_raw = load_from_disk(f'{DATASETS_DIR}/mmlu').to_pandas()

def mmlu_to_unified(row):
    choices  = list(row['choices'])
    ans_idx  = int(row['answer'])
    correct  = choices[ans_idx]
    incorrect = [choices[i] for i in range(len(choices)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{CHOICE_LABELS[i]}) {choices[i]}' for i in range(len(choices))))
    return pd.Series({
        'question': formatted_q, 'question_plain': row['question'],
        'subject': row['subject'], 'best_answer': [correct],
        'correct_answers': [correct], 'incorrect_answers': incorrect,
        'answer_idx': ans_idx, 'choices': choices
    })

mmlu_test_parts, mmlu_icl_parts = [], []
for subject, group in mmlu_raw.groupby('subject'):
    group = group.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_test = min(MMLU_PER_SUBJECT, len(group))
    mmlu_test_parts.append(group.iloc[:n_test])
    if len(group) > n_test: mmlu_icl_parts.append(group.iloc[n_test:])

mmlu     = pd.concat(mmlu_test_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
mmlu_icl = pd.concat(mmlu_icl_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
del mmlu_raw; gc.collect()
print(f'  test={len(mmlu)}Q ({mmlu["subject"].nunique()} subjects) | KB={len(mmlu_icl)}Q (zero overlap)')
print('Datasets ready ✓')

Loading TruthfulQA...
  test=100Q | KB=615Q (zero overlap)
Loading MMLU...
  test=285Q (57 subjects) | KB=1539Q (zero overlap)
Datasets ready ✓


In [4]:
# ── Cell 4: Load model ───────────────────────────────────────────────────────
import gc, torch, psutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

def show_mem():
    ram = psutil.virtual_memory()
    vram = torch.cuda.memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    print(f'  RAM {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f}GB | VRAM {vram:.1f}GB')

gc.collect(); torch.cuda.empty_cache(); show_mem()

print('Loading Mistral-7B...')
if QUANT == '4bit':
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    llm = AutoModelForCausalLM.from_pretrained(f'{MODELS_DIR}/mistral-7b',
                                                quantization_config=bnb, device_map='auto')
else:
    llm = AutoModelForCausalLM.from_pretrained(f'{MODELS_DIR}/mistral-7b',
                                                torch_dtype=torch.float16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(f'{MODELS_DIR}/mistral-7b', padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
show_mem()

print('Loading MiniLM...')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
embed_model = SentenceTransformer(f'{MODELS_DIR}/minilm').to(DEVICE)
show_mem()
print('Models ready!')

  RAM 10.2/47.0GB | VRAM 0.0GB
Loading Mistral-7B...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  RAM 10.9/47.0GB | VRAM 13.5GB
Loading MiniLM...
  RAM 10.9/47.0GB | VRAM 13.6GB
Models ready!


In [5]:
# ── Cell 5: Core utilities ───────────────────────────────────────────────────
import faiss, numpy as np, pandas as pd, torch, gc
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
import mauve as mauve_lib
from tqdm import tqdm

INST_S = '[INST]'
INST_E = '[/INST]'
SYS    = 'You are a truthful expert question-answering bot and should correctly and concisely answer the following question'

def generate(prompts, max_new_tokens=25, num_beams=2):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = llm.generate(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            pad_token_id=tokenizer.eos_token_id,
        )
    return [tokenizer.decode(r[input_len:], skip_special_tokens=True).strip() or 'I have no comment'
            for r in out]

def build_faiss_index(dataset):
    embs = embed_model.encode(dataset['question'].tolist(), show_progress_bar=True, batch_size=64)
    embs = np.array(embs, dtype=np.float32)
    faiss.normalize_L2(embs)
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)
    return idx

def retrieve_top1(query, faiss_idx, kb_dataset):
    q_emb = np.array(embed_model.encode([query], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = faiss_idx.search(q_emb, 1)
    i = idxs[0][0]
    if i < len(kb_dataset):
        return kb_dataset.iloc[i]
    return None

def clean_response(resp):
    for stop in ['\nQuestion:','\nQ:','\n---','\nIncorrect','\nCorrect',
                 '\nVERIFIED','\nExample','\n\n','\nMy initial']:
        if stop in resp: resp = resp[:resp.index(stop)]
    return resp.strip().strip('"').strip("'") or 'I have no comment'

def compute_metrics(generated, references):
    scorer = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1s, r2s, rls, ecss = [], [], [], []
    for gen, refs in zip(generated, references):
        best_r1 = best_r2 = best_rl = 0
        for ref in refs:
            if not ref: continue
            s = scorer.score(ref, gen)
            best_r1 = max(best_r1, s['rouge1'].fmeasure)
            best_r2 = max(best_r2, s['rouge2'].fmeasure)
            best_rl = max(best_rl, s['rougeL'].fmeasure)
        r1s.append(best_r1*100); r2s.append(best_r2*100); rls.append(best_rl*100)
        try:
            embs = embed_model.encode([refs[0], gen])
            ecss.append(float(cosine_similarity([embs[0]], [embs[1]])[0][0])*100)
        except: ecss.append(0.0)
    return np.array(r1s), np.array(r2s), np.array(rls), np.array(ecss)

def compute_mauve(generated, references):
    try:
        refs  = [r[0] if isinstance(r, list) else r for r in references]
        valid = [(g, r) for g, r in zip(generated, refs) if g and r]
        if len(valid) < 10: return 0.0
        gens_v, refs_v = zip(*valid)
        result = mauve_lib.compute_mauve(p_text=list(refs_v), q_text=list(gens_v),
                                          device_id=0, max_text_length=256, verbose=False,
                                          featurize_model_name='gpt2')
        return result.mauve * 100
    except Exception as e:
        print(f'  MAUVE error: {e}'); return 0.0

def compute_accuracy(generated, dataset):
    correct = 0
    for gen, (_, row) in zip(generated, dataset.iterrows()):
        correct_text = row['best_answer'][0] if isinstance(row['best_answer'], list) else row['best_answer']
        ans_idx = int(row.get('answer_idx', -1))
        if ans_idx >= 0:
            label = CHOICE_LABELS[ans_idx]
            if label in gen[:5] or correct_text.lower() in gen.lower(): correct += 1
        else:
            if correct_text.lower() in gen.lower(): correct += 1
    return correct / len(generated) * 100

print('Utilities ready ✓')

Utilities ready ✓


In [6]:
# ── Cell 6: ICL1D+ Core ──────────────────────────────────────────────────────
#
# KEY IDEA (from paper): Use KB questions as contrastive in-context examples.
# For each test question:
#   1. Retrieve the most similar question from KB (top-1)
#   2. Include its CORRECT and INCORRECT answer in the prompt
#   3. Model learns pattern: "this is right, this is wrong, now answer"
#
# This requires KB labels (correct/incorrect answers) — unlike SDCP.
# Prompt format (from paper's rag.py _prompt_template with icl_kb=True):
#   [INST] {SYS}
#   Q: {kb_question}
#   Correct answer: {kb_correct}
#   Incorrect answer: {kb_incorrect}
#   ---
#   Question: {query} Answer: [/INST]

def run_icl1d_plus(test_data, kb_data, dataset_name, params=None):
    if params is None: params = ICL_PARAMS
    print(f'\n=== ICL1D+ on {dataset_name} | test={len(test_data)}Q  KB={len(kb_data)}Q ===')
    print(f'    top_k_docs={params["top_k_docs"]} | num_beams={params["num_beams"]}')

    print('Building KB index...')
    faiss_idx = build_faiss_index(kb_data)

    generated, references = [], []

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc='ICL1D+'):
        query       = row['question']
        best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

        # ── Retrieve top-1 similar question from KB
        kb_hit = retrieve_top1(query, faiss_idx, kb_data)

        if kb_hit is not None:
            # Extract KB example's correct and incorrect answers
            kb_q = kb_hit['question']
            # For MMLU use question_plain to avoid leaking choice labels
            if 'question_plain' in kb_hit.index:
                kb_q = kb_hit['question_plain']

            kb_correct_list   = kb_hit['best_answer']
            kb_correct        = kb_correct_list[0] if isinstance(kb_correct_list, list) and kb_correct_list else str(kb_correct_list)

            kb_incorrect_list = kb_hit['incorrect_answers']
            kb_incorrect      = kb_incorrect_list[0] if isinstance(kb_incorrect_list, list) and kb_incorrect_list else ''

            # ── ICL1D+ prompt: contrastive in-context example
            prompt = (
                f'{INST_S}{SYS}\n'
                f'Q: {kb_q}\n'
                f'Correct answer: {kb_correct}\n'
                f'Incorrect answer: {kb_incorrect}\n'
                f'---\n'
                f'Question: {query} Answer:{INST_E}'
            )
        else:
            # Fallback: no KB hit → plain generation
            prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

        resp  = generate([prompt], max_new_tokens=params['max_gen_tokens'], num_beams=params['num_beams'])
        final = clean_response(resp[0])

        generated.append(final)
        references.append(best_answer)

        if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

    # ── Metrics
    r1, r2, rl, ecs = compute_metrics(generated, references)
    mauve_score     = compute_mauve(generated, references)

    results = {
        'method': f'ICL1D+-{dataset_name}', 'dataset': dataset_name,
        'R1': float(r1.mean()), 'R2': float(r2.mean()),
        'RL': float(rl.mean()), 'ECS': float(ecs.mean()),
        'MAUVE': mauve_score,
        'generated': generated, 'references': references,
    }

    if 'answer_idx' in test_data.columns:
        acc = compute_accuracy(generated, test_data)
        results['Accuracy'] = acc
        print(f'R1={r1.mean():.2f} R2={r2.mean():.2f} RL={rl.mean():.2f} '
              f'ECS={ecs.mean():.2f} MAUVE={mauve_score:.2f} Acc={acc:.1f}%')
    else:
        results['Accuracy'] = 0.0
        print(f'R1={r1.mean():.2f} R2={r2.mean():.2f} RL={rl.mean():.2f} '
              f'ECS={ecs.mean():.2f} MAUVE={mauve_score:.2f}')

    return results

print('ICL1D+ ready ✓')

ICL1D+ ready ✓


In [7]:
# ── Cell 7: Run ICL1D+ on both datasets ──────────────────────────────────────
import time, json

icl_results = {}

print('=' * 60)
print('ICL1D+ — TruthfulQA')
print('=' * 60)
t0 = time.time()
icl_results['ICL1D_TQA'] = run_icl1d_plus(tqa, tqa_icl, 'TruthfulQA')
print(f'  Time: {(time.time()-t0)/60:.1f} min')

print('\n' + '=' * 60)
print('ICL1D+ — MMLU')
print('=' * 60)
t0 = time.time()
icl_results['ICL1D_MMLU'] = run_icl1d_plus(mmlu, mmlu_icl, 'MMLU')
print(f'  Time: {(time.time()-t0)/60:.1f} min')

print('\nICL1D+ complete!')

ICL1D+ — TruthfulQA

=== ICL1D+ on TruthfulQA | test=100Q  KB=615Q ===
    top_k_docs=1 | num_beams=2
Building KB index...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

ICL1D+: 100%|██████████| 100/100 [01:38<00:00,  1.01it/s]


Featurizing p:   0%|          | 0/100 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/100 [00:00<?, ?it/s]

...

ICL1D+: 100%|██████████| 285/285 [05:01<00:00,  1.06s/it]


Featurizing p:   0%|          | 0/285 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/285 [00:00<?, ?it/s]

R1=48.29 R2=37.00 RL=47.74 ECS=61.33 MAUVE=74.85 Acc=57.2%
  Time: 5.1 min

ICL1D+ complete!


WARNING clustering 570 points to 28 centroids: please provide at least 1092 training points


In [8]:
# ── Cell 8: Results + Comparison Table ───────────────────────────────────────

PAPER_BASELINES = {
    'TruthfulQA': {
        'Base (paper)':     {'R1':26.81,'R2':13.26,'RL':23.86,'ECS':56.44,'MAUVE':72.92},
        'ICL1D+ (paper)':   {'R1':30.62,'R2':17.45,'RL':27.79,'ECS':58.96,'MAUVE':73.86},
        'CFR (ours)':       {'R1':38.81,'R2':23.95,'RL':35.74,'ECS':69.35,'MAUVE':12.55},
        'CAFD-LC (ours)':   {'R1':40.66,'R2':25.22,'RL':37.47,'ECS':68.00,'MAUVE':17.97},
        'FACTS-RV (ours)':  {'R1':37.62,'R2':22.22,'RL':34.43,'ECS':68.14,'MAUVE':11.84},
        'SDCP (ours)':      {'R1':34.08,'R2':18.65,'RL':30.93,'ECS':65.80,'MAUVE':12.26},
    },
    'MMLU': {
        'Base (paper)':     {'R1':10.42,'R2':1.90, 'RL':8.91, 'ECS':29.41,'MAUVE':40.51,'Acc':None},
        'ICL1D+ (paper)':   {'R1':26.01,'R2':17.46,'RL':24.90,'ECS':47.04,'MAUVE':37.24,'Acc':None},
        'CFR (ours)':       {'R1':38.56,'R2':28.52,'RL':37.44,'ECS':54.01,'MAUVE':62.73,'Acc':50.88},
        'CAFD-LC (ours)':   {'R1':38.32,'R2':28.27,'RL':37.43,'ECS':54.61,'MAUVE':46.09,'Acc':47.37},
        'FACTS-RV (ours)':  {'R1':26.86,'R2':16.08,'RL':25.18,'ECS':47.94,'MAUVE':46.43,'Acc':45.61},
        'SDCP (ours)':      {'R1':33.77,'R2':24.69,'RL':32.39,'ECS':50.25,'MAUVE':57.68,'Acc':47.02},
    }
}

def print_comparison(dataset_name, icl_key):
    r = icl_results[icl_key]
    is_mmlu = dataset_name == 'MMLU'
    print(f'\n{"="*80}')
    print(f'RESULTS: {dataset_name}')
    print(f'{"="*80}')
    hdr = f'{"Method":<24} {"R1":>6} {"R2":>6} {"RL":>6} {"ECS":>6} {"MAUVE":>7}'
    if is_mmlu: hdr += f' {"Acc":>7}'
    print(hdr)
    print('-' * 80)
    for name, vals in PAPER_BASELINES[dataset_name].items():
        line = (f'{name:<24} {vals["R1"]:>6.2f} {vals["R2"]:>6.2f} '
                f'{vals["RL"]:>6.2f} {vals["ECS"]:>6.2f} {vals["MAUVE"]:>7.2f}')
        if is_mmlu:
            acc_str = f'{vals["Acc"]:>6.1f}%' if vals.get('Acc') else '      —'
            line += acc_str
        print(line)
    print('─' * 80)
    line = (f'{"ICL1D+ (fp16, ours)":<24} {r["R1"]:>6.2f} {r["R2"]:>6.2f} '
            f'{r["RL"]:>6.2f} {r["ECS"]:>6.2f} {r["MAUVE"]:>7.2f}')
    if is_mmlu and 'Accuracy' in r:
        line += f' {r["Accuracy"]:>6.1f}%'
    print(line + ' ◄ NEW')

print_comparison('TruthfulQA', 'ICL1D_TQA')
print_comparison('MMLU', 'ICL1D_MMLU')


RESULTS: TruthfulQA
Method                       R1     R2     RL    ECS   MAUVE
--------------------------------------------------------------------------------
Base (paper)              26.81  13.26  23.86  56.44   72.92
ICL1D+ (paper)            30.62  17.45  27.79  58.96   73.86
CFR (ours)                38.81  23.95  35.74  69.35   12.55
CAFD-LC (ours)            40.66  25.22  37.47  68.00   17.97
FACTS-RV (ours)           37.62  22.22  34.43  68.14   11.84
SDCP (ours)               34.08  18.65  30.93  65.80   12.26
────────────────────────────────────────────────────────────────────────────────
ICL1D+ (fp16, ours)       38.16  23.55  35.33  69.37   17.95 ◄ NEW

RESULTS: MMLU
Method                       R1     R2     RL    ECS   MAUVE     Acc
--------------------------------------------------------------------------------
Base (paper)              10.42   1.90   8.91  29.41   40.51      —
ICL1D+ (paper)            26.01  17.46  24.90  47.04   37.24      —
CFR (ours)            

In [9]:
# ── Cell 9: Save results ──────────────────────────────────────────────────────
from datetime import datetime
import json

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {}
for key, r in icl_results.items():
    summary[key] = {
        'method': r['method'], 'dataset': r['dataset'],
        'R1':  round(r['R1'],  3), 'R2':  round(r['R2'],  3),
        'RL':  round(r['RL'],  3), 'ECS': round(r['ECS'], 3),
        'MAUVE':    round(r['MAUVE'], 3),
        'Accuracy': round(r.get('Accuracy', 0), 2),
    }

out_path = f'{OUTPUT_DIR}/icl1d_plus_summary_{ts}.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

for key, r in icl_results.items():
    df = pd.DataFrame({
        'generated': r['generated'],
        'reference': [x[0] if isinstance(x, list) else x for x in r['references']],
    })
    df.to_csv(f'{OUTPUT_DIR}/{key}_{ts}.csv', index=False, quoting=1)

print(f'Saved to {OUTPUT_DIR}/')
print(json.dumps(summary, indent=2))

Saved to /home/user/RAG_Best_Practices/outputs/
{
  "ICL1D_TQA": {
    "method": "ICL1D+-TruthfulQA",
    "dataset": "TruthfulQA",
    "R1": 38.164,
    "R2": 23.554,
    "RL": 35.331,
    "ECS": 69.373,
    "MAUVE": 17.947,
    "Accuracy": 0.0
  },
  "ICL1D_MMLU": {
    "method": "ICL1D+-MMLU",
    "dataset": "MMLU",
    "R1": 48.291,
    "R2": 37.003,
    "RL": 47.735,
    "ECS": 61.327,
    "MAUVE": 74.845,
    "Accuracy": 57.19
  }
}
